# Import Libraries

In [ ]:
import logging                                  # For logging
from pathlib import Path                        # For path management
from huggingface_hub import snapshot_download   # For fetching the huggingface dataset
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import FeatureUnion
from xgboost import XGBClassifier
import joblib
from imblearn.over_sampling import SMOTE
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV

# Paths
data_path = Path("../data")       # Dataset path
model_path = Path("../model")     # Model path

data_path.mkdir(parents=True, exist_ok=True)
model_path.mkdir(parents=True, exist_ok=True)

# Logging
logging.basicConfig(
    # level = logging.INFO,       # For verbose logging
    level = logging.ERROR,        # For logging only errors
    format = "%(levelname)s: %(message)s"
)

# Fix random seed for reproducibility
np.random.seed(42)

# Initialize dictionary to store application-wide state
state_holder = dict()
logging.info("Initial Setup Process Complete")

# Fetching the dataset

In [5]:
try:
    path = snapshot_download(
        repo_id="sidzzz07/scamshield-dataset",
        repo_type="dataset",
        local_dir=str(data_path)
    )

    logging.info(f"Dataset downloaded to: {path}")

    train_df = pd.read_json(data_path / "train.jsonl", lines=True)
    test_df = pd.read_json(data_path / "test.jsonl", lines=True)
    validation_df = pd.read_json(data_path / "val.jsonl", lines=True)

except FileNotFoundError:
    logging.error("Dataset is not found, please check the dataset path and try again")
except Exception as e:
    logging.error(f"Failed to load dataset {e}")

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 239.48it/s]


# Training Dataset analysis

Showing random 5 rows from the training dataset

In [6]:
train_df.sample(5)

,id,text,language,source_dataset,is_scam,head1_social_engineering,head2_scam_intent,split_group,metadata
42976,real_076848,Hello sir Congratulations! Aapne ₹40 lakh ka l...,Hinglish,ysangam/Indian_Cyber_Scam_Hinglish,1,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Lottery / Prize,real_ysangam_indian_cyber_scam_hinglish,"{'has_url': False, 'has_phone': False, 'has_ot..."
36830,real_030308,ok ill be waiting,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
52659,real_025414,Cafe Iguana #01-03 Riverside point 30 merchan...,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
14804,real_071896,Lol probably everyone finished their paper exc...,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
21475,real_067806,Erm u meet her help me ask eve whether she hav...,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."


Showing the dataset columns

In [7]:
train_df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 69286 entries, 0 to 69285
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   id                        69286 non-null  str   
 1   text                      69286 non-null  str   
 2   language                  69286 non-null  str   
 3   source_dataset            69286 non-null  str   
 4   is_scam                   69286 non-null  int64 
 5   head1_social_engineering  69286 non-null  object
 6   head2_scam_intent         69286 non-null  str   
 7   split_group               69286 non-null  str   
 8   metadata                  69286 non-null  object
dtypes: int64(1), object(2), str(6)
memory usage: 4.8+ MB


In [ ]:
language = train_df["language"].value_counts()

px.pie(
    names=language.keys(),
    values=language.values,
    title="Partition of languages in the training dataset",
)

In [61]:
scam_intent = train_df["head2_scam_intent"].value_counts()

px.bar(
    x=scam_intent.keys(),
    y=scam_intent.values,
    title="Partition of scam intent in the trianing class",
    text_auto=".0f"
)

In [44]:
is_scam = train_df["is_scam"].value_counts()

px.pie(
    names=is_scam.keys(),
    values=is_scam.values,
    title="Partition of classes",
    subtitle="(0 -> Not Spam, 1 -> Spam)",
)

## Key insights

1) Majority of the writers of the message uses English as their communication language.
2) Out of all the messages in the training dataset, most of messages are legitimate.
3) Out of all the scam messages, most of them are based on telemarketing. This reflects that fraudulent messages uses promotional offers, unsolicit marketing, fake rewards or deceptive sales tactics as a common way of scamming.
2) The dataset is imbalanced based on the partition of 36.2% of the class is spam and 63.8% of the data are not spam.

# Model Training

In [55]:
X_train = train_df["text"]
X_test = test_df["text"]

y_scan_train = train_df["is_scam"]
y_scan_test = test_df["is_scam"]

y_social_train = pd.DataFrame(
    train_df["head1_social_engineering"].tolist(),
    index=train_df.index
)
y_social_test = pd.DataFrame(
    test_df["head1_social_engineering"].tolist(),
    index=test_df.index
)

y_intent_train = train_df["head2_scam_intent"]
y_intent_test = test_df["head2_scam_intent"]

intent_encoding = LabelEncoder()
y_intent_train = intent_encoding.fit_transform(y_intent_train)
y_intent_test = intent_encoding.transform(y_intent_test)